=========================================================================================
Simplified Time-Period Mobility Prediction Model
=========================================================================================

[Core Prediction Logic: The 7-Day Human Cycle]
This model is built on the assumption that human mobility is highly repetitive on a 
weekly basis. The best way to predict where a user will be at a specific time on a 
specific day is to look at where they most frequently were at that EXACT "Weekday + Time" 
in the past.

The algorithm works in 3 steps:
1. Spatial Stationarity & Imputation (The ffill)
   Real-world trajectory data is "sparse" (only recorded when a person moves). We forcefully 
   build a complete time grid of 60 days × 48 time bins = 2880 rows. Using forward fill 
   (ffill), if a time bin lacks data, we assume the user "stayed where they were."
   This step is crucial: it gives locations with long "stay durations" (like home or work) 
   a massive weight advantage in the final voting process.

2. Mining Historical Patterns (Highest Frequency Voting)
   We attach a "Weekday" label to these 2880 filled bins. For all 336 possible state 
   combinations (7 Days × 48 Bins), we count the frequencies and extract the single most 
   frequently visited (x, y) coordinate as the "Standard Answer" for that state.

3. Direct Mapping (Zero Fallback Needed)
   Because Step 1 perfectly filled all the time gaps in the past 60 days, our "Standard 
   Answers" for the 336 states are 100% complete. When predicting Days 60-74, we simply 
   use the future "Weekday + Time" to look up the answer. No fallback or cascading logic 
   is required.

[Engineering Architecture: OOM-Proof Streaming]
Reading massive CSV files directly will cause Out-of-Memory (OOM) crashes.
This script uses a "Sliding Window Chunking + Buffer Stitching" architecture:
- It only loads 500,000 rows into memory at a time.
- It checks the last user in the current chunk. If their data is cut off (half in this 
  chunk, half in the next), it slices that user out and puts them in a 'Buffer'.
- It processes the remaining complete users, appends their predictions directly to the 
  hard drive (to_csv(mode='a')), and immediately frees the memory.
- With this architecture, whether your source file is 1GB or 100GB, RAM usage remains 
  stable at just a few megabytes.
=========================================================================================

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
output_path = 'MathV1.csv'

# ==========================================
# Core Prediction Engine: Extract and Map Single User's Historical Patterns
# ==========================================
def process_single_user(user_df):
    uid = user_df['uid'].iloc[0]
    
    # 1. Filter the first 60 days as the training set. If there are multiple records 
    # at the same time, keep the last one (representing the final settled location).
    train_df = user_df[user_df['d'] <= 59].drop_duplicates(subset=['d', 't'], keep='last')
    if train_df.empty: return pd.DataFrame()

    # 2. Build a complete time grid of 2880 bins and apply forward fill (Stationary Assumption)
    full_index = pd.MultiIndex.from_product([[uid], range(60), range(48)], names=['uid', 'd', 't'])
    train_full = train_df.set_index(['uid', 'd', 't']).reindex(full_index)
    train_full = train_full.ffill().bfill().reset_index()
    train_full['weekday'] = train_full['d'] % 7
    
    # 3. Core calculation: Find the single most frequent (x, y) coordinate for all 336 combinations
    best_prob = train_full.groupby(['weekday', 't', 'x', 'y']).size().reset_index(name='count')\
                          .sort_values('count', ascending=False).drop_duplicates(['weekday', 't'])

    # 4. Generate the prediction grid for the next 15 days (Days 60-74)
    pred_index = pd.MultiIndex.from_product([[uid], range(60, 75), range(48)], names=['uid', 'd', 't'])
    pred_df = pd.DataFrame(index=pred_index).reset_index()
    pred_df['weekday'] = pred_df['d'] % 7

    # 5. Direct mapping: Use the future 'weekday + time' to look up the historical 'standard answer'
    ans = pd.merge(pred_df, best_prob[['weekday', 't', 'x', 'y']], on=['weekday', 't'], how='left')
    
    return ans[['uid', 'd', 't', 'x', 'y']].astype(int)

# ==========================================
# Execution Engine: Chunk Reading & Buffer Stitching
# ==========================================
pd.DataFrame(columns=['uid', 'd', 't', 'x', 'y']).to_csv(output_path, index=False)
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}
buffer_df = pd.DataFrame()

TARGET_USERS = 10000 
user_count, stop_reading = 0, False

print(f"Starting simplified calculation for {TARGET_USERS} users...")

for chunk in pd.read_csv(path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=500000):
    if stop_reading: break
        
    # Stitch the 'tail user' from the previous chunk to the head of the current chunk
    chunk = pd.concat([buffer_df, chunk], ignore_index=True)
    unique_uids = chunk['uid'].unique()
    
    # If the entire chunk belongs to a single user, put the whole chunk into the buffer
    if len(unique_uids) == 1:
        buffer_df = chunk; continue
        
    # The last user's data might be incomplete. Separate it as the new 'tail' for the next round
    last_uid = unique_uids[-1]
    complete_data, buffer_df = chunk[chunk['uid'] != last_uid], chunk[chunk['uid'] == last_uid]
    
    # Independently predict for all users with complete data in the current chunk
    for uid, user_data in complete_data.groupby('uid'):
        if user_count >= TARGET_USERS: stop_reading = True; break
            
        user_pred = process_single_user(user_data)
        if not user_pred.empty:
            # Compute and save immediately to the hard drive, freeing up RAM
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
            
        user_count += 1
        if user_count % 1000 == 0: print(f"Processed {user_count} / {TARGET_USERS} users...")

# After the loop ends, process the final user remaining in the buffer
if not stop_reading and not buffer_df.empty and user_count < TARGET_USERS:
    for uid, user_data in buffer_df.groupby('uid'):
        user_pred = process_single_user(user_data)
        if not user_pred.empty: user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1

print(f"\nTask Complete! Results saved to: {output_path}")

Starting simplified calculation for 10000 users...
Processed 1000 / 10000 users...
Processed 2000 / 10000 users...
Processed 3000 / 10000 users...
Processed 4000 / 10000 users...
Processed 5000 / 10000 users...
Processed 6000 / 10000 users...
Processed 7000 / 10000 users...
Processed 8000 / 10000 users...
Processed 9000 / 10000 users...
Processed 10000 / 10000 users...

Task Complete! Results saved to: MathV1.csv


=============================================================================
Model Evaluation: Compare Predictions against Ground Truth (Days 60-74)
=============================================================================
This script evaluates the accuracy of the Markov prediction model.
It loads the generated predictions, then streams the original dataset to extract
only the TRUE trajectory records for days 60-74 for the evaluated users.
Finally, it merges them on [uid, d, t] and calculates the Top-1 Accuracy.
=============================================================================


In [7]:
# 1. File paths (Make sure these match your actual files)
pred_path = 'MathV1.csv'
orig_path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'

print("1. Loading predictions into memory...")
# Load the predictions we just generated
pred_df = pd.read_csv(pred_path)
target_uids = pred_df['uid'].unique()
print(f"-> Successfully loaded predictions for {len(target_uids)} users.")

print("\n2. Extracting Ground Truth (Days 60-74) from massive dataset...")
gt_chunks = []
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}

# Stream original data to prevent Memory Crash
for chunk in pd.read_csv(orig_path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=1000000):
    
    # Filter 1: Only keep Days 60 to 74
    chunk = chunk[(chunk['d'] >= 60) & (chunk['d'] <= 74)]
    
    # Filter 2: Only keep users that we actually predicted
    chunk = chunk[chunk['uid'].isin(target_uids)]
    
    # Remove duplicates just in case
    chunk = chunk.drop_duplicates(subset=['uid', 'd', 't'], keep='last')
    
    if not chunk.empty:
        gt_chunks.append(chunk)

# Combine all valid chunks into the final Ground Truth dataframe
gt_df = pd.concat(gt_chunks, ignore_index=True)
print(f"-> Extracted {len(gt_df)} real trajectory records for evaluation.")

print("\n3. Comparing Predictions vs. Ground Truth...")
# Merge Ground Truth and Predictions on UID, Day, and Time Bin
# suffix _true for actual data, _pred for our model's data
eval_df = pd.merge(gt_df, pred_df, on=['uid', 'd', 't'], suffixes=('_true', '_pred'))

# A prediction is correct ONLY if both X and Y coordinates match exactly
eval_df['is_correct'] = (eval_df['x_true'] == eval_df['x_pred']) & (eval_df['y_true'] == eval_df['y_pred'])

# Calculate Accuracy (Correct predictions / Total evaluated records)
accuracy = eval_df['is_correct'].mean() * 100
total_evaluated = len(eval_df)
correct_count = eval_df['is_correct'].sum()

print("==================================================")
print("📊 EVALUATION RESULTS")
print("==================================================")
print(f"Total Evaluated Records: {total_evaluated}")
print(f"Correct Predictions:     {correct_count}")
print(f"Overall Accuracy:        {accuracy:.2f} %")
print("==================================================")

# Optional: View where the model failed
# errors = eval_df[eval_df['is_correct'] == False]
# print(errors.head())

1. Loading predictions into memory...
-> Successfully loaded predictions for 10000 users.

2. Extracting Ground Truth (Days 60-74) from massive dataset...
-> Extracted 2470704 real trajectory records for evaluation.

3. Comparing Predictions vs. Ground Truth...
📊 EVALUATION RESULTS
Total Evaluated Records: 2470704
Correct Predictions:     613138
Overall Accuracy:        24.82 %
